# U5-L1：语音识别

**课程目标**

1. 能区分录音与语音识别的输入、输出：录音得到声音的采样数据，语音识别将语音转成文字；转成文字后还需要结合语境理解意思。
2. 能说明采样率、声道数和采样格式的含义，并用它们估算 PCM 数据量与音频长度。
3. 能说明播放队列如何暂存声音块，比较不同队列容量下的写入等待与播放现象。
4. 能比较整句识别与流式识别的适用场景，并启动、观察和停止流式识别任务。


## 一、本课要解决的问题

前面我们用视觉和运动接口，让机器人发现来宾、转向来宾，并尝试接近来宾。

如果还要与来宾交流，程序就需要取得来宾说话的内容。本课先研究麦克风采到什么，再让机器人把一句话转成文字。

> **装上麦克风后，机器人怎样把声音变成文字？**


## 二、麦克风采集的声音

麦克风可以比作机器人的耳朵。它感知空气中声压的变化，音频采集系统再把这些变化转换为数字数据。说话声、脚步声、周围的噪声，都可能被采集下来。

程序通过录音接口取得这些数据。麦克风本身不会判断一句话说了什么。


### 2.1 麦克风采集声音数据的原理

声音在空气中传播时会引起声压变化，麦克风把这种变化转为电信号。音频采集系统每隔一小段时间量一次信号，按顺序记录数值，就得到一段音频数据。这种按时间取数的过程叫**采样**。

读懂这段数据，需要看三个参数：

- **采样率（sample_rate）**：每个声道一秒采样多少次。16000 Hz 表示每个声道每秒采样16000次，是语音处理中常见的设置。
- **声道数（channels）**：同时记录几路声音数据。单声道记录一路；三声道记录三路，每一路都有自己的采样数值。
- **采样格式（sample_format）**：每个采样数值怎么存。`S16LE` 表示用16位有符号整数、小端顺序存储，每个数占2字节。

先按**单声道**来算：一秒采样16000次、每次记2字节，共 `16000 × 1 × 2 = 32000` 字节。

本课使用的K1有环形三麦克风阵列，后面的采音代码设置 `channels=3`。同样的一秒声音需要记录三路数据，因此是 `16000 × 3 × 2 = 96000` 字节。声道增加不会把这段声音变长，增加的是同一段时间内记录的数据量。

对于本课使用的未压缩 `S16LE PCM` 数据：

```text
数据量（字节） = 时长（秒） × 采样率 × 声道数 × 2
音频长度（秒） = 数据量（字节） ÷（采样率 × 声道数 × 2）
```

这个公式计算音频采样数据，不包含WAV等文件的文件头，也不适用于MP3等压缩数据。采样数据记录的是声音变化，其中还没有文字。


### 2.2 实践：调用麦克风采集一段声音

1. 调用 `start_recording()` 开始录音，采到的数据暂存在内存里。
2. 调用 `stop_recording()` 停止录音，返回一个 `AudioData` 对象，包含音频数据及其格式信息。
3. 需要保存为文件时，再调用这个对象的 `save(path)`。

如果已经调用 `start_recording()`，但尚未调用 `stop_recording()`，再次开始录音会出错。可用 `is_recording()` 检查当前状态；若上一次还在录音，先停止，再运行下一次录音。

录制过程中，`get_recording_duration()` 可查询已经录了多少秒。


**STEP 01：** 连接机器人，确认音量

- `BoosterRobot.audio_manager` 用于获取与当前 BoosterRobot 绑定的音频管理器实例，作为访问音频子系统接口的入口。
- `BoosterAudioManager.get_system_volume()` 查询机器人系统输出音量。


In [35]:
from boosteros.robots.booster import BoosterRobot

robot = BoosterRobot()

print("已连接：", robot.robot_info.model, robot.robot_info.serial_number)
robot.audio_manager.set_system_volume(0.5)  # 将音量调小一点
print("当前音量：", robot.audio_manager.get_system_volume())


[Booster OS] [boosteros.sdk] [INFO] Starting Booster SDK Local Adapter (mode=local, network_interface=default, virtual_robot_name=default).
[Booster OS] [boosteros.sdk] [INFO] Booster SDK Local Adapter started (mode=local, network_interface=default).


ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024


[Booster OS] [boosteros.robot] [INFO] Fetching camera configuration from API...
[Booster OS] [boosteros.robot] [INFO] Using cameras ['head'] from API, default camera is 'head'.
[Booster OS] [boosteros.BoosterHandGuidingManager] [INFO] BoosterHandGuidingManager initialized with default_record_file=/opt/booster/Gait/bin/joint_position.txt, record_archive_dir=~/.boosteros/hand_guiding
[Booster OS] [boosteros.ros] [INFO] [/boosteros_ros2_adapter_e3782282] Booster ROS 2 Adapter initialized. Cameras: head(rgb=/boostercamera/head/rgb, depth=/boostercamera/head/depth). Default camera: head.
[Booster OS] [boosteros.ros] [INFO] [/boosteros_ros2_adapter_e3782282] Booster ROS 2 Adapter started with 4 threads.
[Booster OS] [boosteros.robot] [INFO] Initializing audio manager & speech runtime...
[Booster OS] [boosteros.BoosterAudioManager] [INFO] _BoosterAudioRuntime initialized
[Booster OS] [boosteros.BoosterAudioManager] [INFO] BoosterAudioManager initialized
[Booster OS] [boosteros.BoosterSpeechRu

ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024

**STEP 02：** 录一段话

<font color="red">提示：这一格开始录音后，对着机器人头部的方向说一句话，不用喊，正常音量就行。3 秒后自动结束。</font>

In [36]:
import time

robot.audio_manager.start_recording(channels=3)  # 声道数设置为3，因为K1是环形 3 麦克风阵列
print("正在录音，请说一句话")

for i in range(3):
    time.sleep(1)
    print(f"  已录 {robot.audio_manager.get_recording_duration():.3f} 秒")

audio = robot.audio_manager.stop_recording()
print("录音结束，还在录吗：", robot.audio_manager.is_recording())

ChannelSubscriber::InitChannel: setting reliability: 0, queue_capacity: 64
ChannelSubscriber::InitChannel: setting reliability: 正在录音，请说一句话
0, queue_capacity: 64
  已录 0.000 秒
  已录 1.600 秒
  已录 2.368 秒
录音结束，还在录吗： False


### 2.3 录制的音频数据里有什么

`stop_recording()` 返回的 `AudioData` 把声音数据和格式信息放在一起：

- `data`：音频字节数据，本次录音为PCM数据；
- `duration`：音频时长，类型是 `Duration`，用 `duration.seconds` 取秒数；
- `sample_rate`、`channels`、`sample_format`：采样率、声道数、采样格式。

调用 `audio.save(path)` 可以保存到文件，`path` 是文件路径字符串，支持 `.wav`、`.pcm` 或 `.mp3` 后缀。本课保存为WAV文件。


**STEP 03：** 读一读这段数据

In [7]:
import os

print("音频对象：", audio)

print("时长：", audio.duration)
print("时长的秒数：", round(audio.duration.seconds, 2), "秒")
print("采样率：", audio.sample_rate, "　声道数：", audio.channels)
print("采样格式：", audio.sample_format)
print("数据长度：", len(audio.data), "字节")

save_path = os.path.join(os.getcwd(), "my_voice.wav")
audio.save(save_path)
print(f"已保存到{save_path}")

[Booster OS] [boosteros.AudioData] [WARNING] Downmixed 3-channel audio to 2-channel stereo for WAV playback compatibility; the individual channel data is lost. Save as .pcm to keep every channel intact.


音频对象： Audio Data @ 1792640383.321
  Coding:      pcm
  Sample Fmt:  S16LE
  Sample Rate: 16000 Hz
  Channels:    3
  Duration:    3.07 s
时长： <Duration 3.072000s>
时长的秒数： 3.07 秒
采样率： 16000 　声道数： 3
采样格式： S16LE
数据长度： 294912 字节
已保存到/home/booster/Workspace/XC/my_voice.wav


### 2.4 放出来听听

调用 `robot.play_sound(audio, volume=0.5)` 可以启动播放，`volume` 要用参数名指定，范围为 `0.0`～`1.0`。

本课会使用两种常见输入：内存里的 `AudioData`，或已保存的音频文件路径。下面先播放刚保存的文件。

`play_sound()` 会立即返回任务句柄，声音在后台播放。调用这个句柄的 `wait()`，当前代码才会等待播放任务结束。声音可以在没有 `wait()` 时播放；等待的作用是让后续操作按顺序进行。还要查看任务状态，才能区分播放成功、失败或取消。


**STEP 04：** 把刚录的放一遍

In [8]:
playback = robot.play_sound(save_path)
playback_status = playback.wait()
if playback_status == "SUCCEEDED":
    print(f"文件 {save_path} 播放完成")
else:
    print("播放未成功，状态：", playback_status, "；错误：", playback.error)


ChannelSubscriber::InitChannel: setting reliability: 0, queue_capacity: 64
文件/home/booster/Workspace/XC/my_voice.wav播放结束


## 三、声音数据的分块读取

刚才先录3秒，再取得整段声音。如果要持续接收声音，一直等到录音结束才处理，就无法及时显示字幕。

可以把持续输入的声音比作水龙头流出的水：用杯子一杯杯接，每接好一杯就处理一杯。程序也可以每次取得一小块声音，逐块处理，不必等整段录音结束。


### 3.1 分块读取

`record_stream()` 返回一个迭代器。在 `for` 循环中，每次取得一个 `AudioData` 声音块，就像接好的一杯水。

`chunk_bytes` 指定每块的字节数。本课保持 `sample_rate=16000`、`channels=3`、`sample_format="S16LE"`，每秒共96000字节。因此，3200字节约对应 `3200 ÷ 96000 ≈ 0.033` 秒的声音。这里要把三路数据一起计算，不能套用单声道的0.1秒。

`stop_event` 接收一个停止标志。下面用 `threading.Event()` 建立标志，调用它的 `set()` 通知采音循环停止。本课只把它当作停止开关使用。

块的实际长度可用 `len(chunk.data)` 查看。将所有块的字节数相加，再按格式换算，得到的是这些数据所包含的音频长度；程序运行时间还包括启动、传输和输出等时间。


**STEP 05：** 一块一块地读麦克风

这一格取得20块声音。按每块3200字节估算，音频长度约0.67秒；它与代码格运行时间可能不同。声音片段很短，说一个字或轻拍一下手，观察块数和字节数。


In [10]:
import threading

stop_flag = threading.Event()
count = 0
total_bytes = 0

for chunk in robot.audio_manager.record_stream(channels=3, chunk_bytes=3200, stop_event=stop_flag):
    count += 1
    total_bytes += len(chunk.data)
    if count % 5 == 0:
        print(f"第 {count} 块，本块 {len(chunk.data)} 字节")
    if count >= 20:
        stop_flag.set()

audio_seconds = total_bytes / (16000 * 3 * 2)
print(f"读完，共 {count} 块、{total_bytes} 字节，约含 {audio_seconds:.3f} 秒音频")


ChannelSubscriber::InitChannel: setting reliability: 0, queue_capacity: 64
第 5 块，3200 字节
第 10 块，3200 字节
第 15 块，3200 字节
第 20 块，3200 字节
读完，共 20 块，每块 3200 字节


### 3.2 流式播放与队列

`record_stream()` 一块块读入声音，`play_stream()` 则允许一块块写入声音并播放。

- `play_sound()`：把已有的整段音频交给它播放。
- `play_stream()`：创建播放流，再用 `write(chunk)` 逐块交入格式一致的音频。

继续用接水作比喻：输入端负责把一杯杯水送来，播放端按自己的速度取走。送来得快了，需要先放在一排暂存位置上。这些位置就是**队列**，每个位置暂存一个声音块。

- `queue_size`：队列最多暂存几块，默认64，必须大于0。
- `overflow`：队列满后的处理方式。`"block"` 让当前写入等待空位；`"drop_oldest"` 丢弃队列里最旧的块；`"drop_newest"` 丢弃新送来的块。

一次写入返回，只能说明这次提交已处理，声音可能仍在队列里等待播放。


### 3.3 队列的深浅

暂存位置多，可以容纳更多声音块；位置少，较快就会排满。

- **浅队列**：暂存量较少。选择 `overflow="block"` 时，输入较快就可能需要等待；选择丢弃策略时，可能损失声音块。
- **深队列**：可以吸收更大的一阵输入。若输入持续快于播放，仍会排满；已经排在前面的声音越多，新块等待播放的时间通常越长。

例如本课每块约0.033秒，队列里若已有30块待播放，单是这些声音就约有1秒。队列容量大，并不表示每次都会填满，也不能仅凭容量推算实际延迟。

队列容量和溢出策略共同影响等待与数据完整性。这类选择属于 **QoS（服务质量）** 的设置。实时字幕和完整录音对“及时”和“保留数据”的要求可能不同，要按任务选择。


**STEP 06：** 采集一段声音，把声音块存到列表

本次先录后放，便于分别观察采集和播放。下面存60块，按当前参数约含2秒音频，可以说一句简短的话。录音结束后，用同一批块比较两种队列容量。


In [11]:
import threading

print("开始录制，请说一句话")
stop_flag = threading.Event()

chunks = []
count = 0
for chunk in robot.audio_manager.record_stream(
    channels=3, chunk_bytes=3200, stop_event=stop_flag
):
    chunks.append(chunk)
    count += 1
    if count >= 60:                   # 三声道下约含2秒音频
        stop_flag.set()
        break

total_bytes = sum(len(chunk.data) for chunk in chunks)
audio_seconds = total_bytes / (16000 * 3 * 2)
print(f"录制完成，共 {count} 块、{total_bytes} 字节，约含 {audio_seconds:.3f} 秒音频")


开始录制，请说一句话
ChannelSubscriber::InitChannel: setting reliability: 0, queue_capacity: 64
录制完成，共 60 块


**STEP 07：** 用同一批声音块比较队列容量

先用 `queue_size=4` 运行下一格，确认播放任务结束后，再改成64重跑；录音块保持不变，`overflow` 都设为 `"block"`。

记录全部块写入所用的总时间，并听声音是否完整、有无卡顿。小队列通常更容易出现明显的写入等待；大队列可能较快接收完这批块，但接收完不等于播放完。每次函数调用都有开销，不能把微小的非零耗时直接当作队列阻塞。

写完后按以下顺序收尾：

```python
stream.close()                 # 后面不再写入，已排队的声音继续播放
status = stream.wait(timeout=10.0)  # 等待整个播放任务结束，或超时返回当前状态
```

`close()` 可以比作通知播放端“这是最后一杯，后面没有了”，不表示把已排队的声音倒掉。若先 `wait()`，流仍可能等待后续输入，不能据此确认整段播放完成。需要立即中止并清空队列时，用 `cancel()`。这些含义见SDK的[播放流句柄说明](https://docs.booster.tech/zh-CN/docs/developer-guide/booster-os-python-sdk/common-data-types/tasks-and-subscriptions/audio-playback-stream-handle/)。

`wait()` 返回后，`SUCCEEDED` 才表示播放成功；`FAILED`、`CANCELLED` 分别表示失败和取消；其余状态表示还没有结束。超时不会自动取消任务。若仍未结束，可运行 `stream.cancel()`，再用 `stream.wait(timeout=5.0)` 检查，暂不创建下一条播放流。

下面保留原真机实验的两组输出，分别对应容量4和64，仅供比较写入耗时。原输出末行“播放结束”是旧程序直接打印的，不能作为任务成功的证据；修订后的代码会显示等待返回的真实状态。

```text
开始写入播放流...
            已写入 10/60 块，当前耗时 33.8ms
            已写入 20/60 块，当前耗时 34.9ms
            已写入 30/60 块，当前耗时 38.0ms
            已写入 40/60 块，当前耗时 49.5ms
            已写入 50/60 块，当前耗时 31.3ms
            已写入 60/60 块，当前耗时 37.1ms
        所有音频块已写入播放流，总计耗时 1.850522756576538
        播放结束
```

```text
开始写入播放流...
            已写入 10/60 块，当前耗时 0.0ms
            已写入 20/60 块，当前耗时 0.0ms
            已写入 30/60 块，当前耗时 0.0ms
            已写入 40/60 块，当前耗时 0.0ms
            已写入 50/60 块，当前耗时 0.0ms
            已写入 60/60 块，当前耗时 0.0ms
        所有音频块已写入播放流，总计耗时 0.0007965564727783203
        播放结束
```


In [34]:
import time

queue_size = 4  # 第一组为4；播放结束后改为64，再运行本格
terminal_states = ("SUCCEEDED", "FAILED", "CANCELLED")

if "stream" in globals() and stream.wait(timeout=0.0) not in terminal_states:
    print("上一条播放流尚未结束。请保留 stream，先取消并等待结束，再重试。")
else:
    stream = robot.audio_manager.play_stream(channels=3, queue_size=queue_size, overflow="block")
    print("开始写入播放流...")
    start_time = time.perf_counter()
    try:
        for i, chunk in enumerate(chunks):
            write_start = time.perf_counter()
            stream.write(chunk)
            write_time = time.perf_counter() - write_start
            if (i + 1) % 10 == 0:
                print(f"  已写入 {i + 1}/{len(chunks)} 块，本次写入耗时 {write_time * 1000:.1f} ms")
        print(f"全部块写入耗时：{time.perf_counter() - start_time:.3f} 秒")
    finally:
        stream.close()  # 结束输入，排队的音频继续播放
    playback_status = stream.wait(timeout=10.0)
    if playback_status == "SUCCEEDED":
        print("播放完成，状态：", playback_status)
    elif playback_status in terminal_states:
        print("播放未成功，状态：", playback_status)
    else:
        print("等待超时，播放任务尚未结束，状态：", playback_status)
        print("保留 stream 继续检查；暂不运行下一组。")


[Booster OS] [boosteros.BoosterAudioManager] [WARNING] Downmixing 3-channel audio to mono for playback; the SDK does not play multi-channel PCM streams correctly. Individual channel data is averaged.


开始写入播放流...
  已写入 10/60 块，当前耗时 33.8ms
  已写入 20/60 块，当前耗时 34.9ms
  已写入 30/60 块，当前耗时 38.0ms
  已写入 40/60 块，当前耗时 49.5ms
  已写入 50/60 块，当前耗时 31.3ms
  已写入 60/60 块，当前耗时 37.1ms
所有音频块已写入播放流，总计耗时 1.850522756576538
播放结束


## 四、从声音到文字

录音得到了声音的采样数据。如果里面有人说话，程序还需要把语音转成文字，才能继续处理提问的内容。


### 4.1 语音识别

把语音转成文字叫**语音识别**，英文缩写 ASR。

可以把它理解为给录音配字幕：模型根据声音的特征和上下文，生成可能对应的文字。模型通常通过大量语音和文字资料学习这种对应关系；不同模型的内部做法会有差别。

在本课中，主要看接口的输入和输出：

- 输入：包含人类语音的音频；
- 输出：识别出的文字，可能存在误识别。

写出“几点开门”只完成了转写。要知道来宾在询问开放时间，并给出回答，还需要后续处理。


### 4.2 流式与整句

按照音频交入和文字返回的方式，可以比较两种常见用法：

| 方式 | 怎样取得结果 | 应用中的需要 |
|---|---|---|
| 整句（整段）识别 | 一句话或一段录音准备好后，整段交入，再取得文字 | 已录好的语音留言转写，可以等完整录音准备好再处理 |
| 流式识别 | 声音持续分块交入，在说话过程中陆续返回文字 | 会议实时字幕、边说边输入，希望讲话过程中就看到文字 |

两种方式都把音频转成文字。同一个产品也可以组合使用：例如语音助手先持续接收声音，在判断用户说完后再整理结果。不能只凭“智能音箱”或“会议软件”的名称判断内部用了哪种接口。

流式识别很像一边听一边写字幕，前面显示的文字可能随着后面的声音修正。本课 `recognize_stream()` 的回调只给出 `text`，没有“本句最终结果”的标志，因此不能把最后一次回调直接当作已经确认的完整句子。


## 五、实验：把一句话变成屏幕上的字

⚠️ **注意**：本课不让机器人走动，只用到麦克风与扬声器。录音时场地安静一些，效果会更好；机器人的音量从小往大调到一个合适的值，不要太小听不到，也不要过大影响别人。

同一句话识别两次，看两次识别出来的结果是不是一样。

### 5.1 第一次识别

`BoosterOS SDK` 中提供的流式识别接口是 `recognize_stream()`：它让机器人用麦克风持续识别，参数 `on_result` 指定一个回调函数，每识别出一段文字就调用一次。回调函数只接收一个参数 `text`，就是刚识别出来的那段文字。

这是持续任务，不以一句话识别完成为结束条件，也可能因错误退出。本实验留6秒观察窗口，再调用 `cancel()` 请求停止，并用 `wait(timeout=5.0)` 等待收尾。

`cancel()` 的返回值只表示取消请求是否提交成功。`done()` 为 `True` 才说明任务已结束，成功、失败、取消都属于结束状态；`running()` 为假还可能是等待或取消中。若仍未结束，保留句柄继续检查，不能覆盖它另开一次识别。

**STEP 08：** 说一句话，看输出的文字


In [19]:
from boosteros.brain import Speech

speech = Speech(robot)

def on_text(text):
    print("识别到：", text)

if "task" in globals() and not task.done():
    print("上一项语音任务尚未结束，状态：", task.status)
    print("先运行 task.cancel()，再运行 task.wait(timeout=5.0)，确认 task.done() 为 True 后重试。")
else:
    task = speech.recognize_stream(on_result=on_text)
    print('开始识别，请说一句话')
    try:
        time.sleep(6)  # 留出6秒供观察文字回调，不代表识别一句话需要6秒
    finally:
        accepted = task.cancel()
        print("取消请求已提交：", accepted)
        task.wait(timeout=5.0)
    print("当前状态：", task.status, "；是否已结束：", task.done())
    if not task.done():
        print("任务尚未结束，保留 task，继续检查；暂不启动下一次识别或对话。")
    elif task.status == "FAILED":
        print("任务失败：", task.error)


开始识别，请说一句话
识别到： 人工
识别到： 人工智能
识别到： 人工智能
识别到： 人工智能
识别结束，任务状态： CANCELLED


### 5.2 第二次识别

把刚才那句话再说一遍。识别结果依赖于输入的声音，每次录制的声音都是不一样的——比如语速、音量、周围环境噪音等，这些都会影响结果。

**STEP 09：** 同样的话再说一遍

In [20]:
if "task" in globals() and not task.done():
    print("上一项语音任务尚未结束，状态：", task.status)
    print("先运行 task.cancel()，再运行 task.wait(timeout=5.0)，确认 task.done() 为 True 后重试。")
else:
    task = speech.recognize_stream(on_result=on_text)
    print('再说一遍刚才那句话')
    try:
        time.sleep(6)  # 留出6秒供观察文字回调，不代表识别一句话需要6秒
    finally:
        accepted = task.cancel()
        print("取消请求已提交：", accepted)
        task.wait(timeout=5.0)
    print("当前状态：", task.status, "；是否已结束：", task.done())
    if not task.done():
        print("任务尚未结束，保留 task，继续检查；暂不启动下一次识别或对话。")
    elif task.status == "FAILED":
        print("任务失败：", task.error)


再说一遍刚才那句话
识别到： 人工
识别到： 人工智能
识别到： 人工智能
识别结束，任务状态： CANCELLED


## 六、观察与记录

### 6.1 录到的这段数据

| 项目 | 记录 |
| :--- | :--- |
| 音频时长（秒） | 　　　　 |
| 采样率 / 声道数 / 采样格式 | 　　　　 |
| PCM数据长度（字节） | 　　　　 |
| 按“时长 × 采样率 × 声道数 × 2”估算（字节） | 　　　　 |

### 6.2 同一批声音块的播放对照

声音块数：________；总字节数：________；换算的音频长度：________秒。

| 队列容量 | 全部块写入耗时（秒） | 是否观察到明显写入等待 | 声音是否完整、有无卡顿 | 等待返回的状态 |
|---|---|---|---|---|
| 4 |  |  |  |  |
| 64 |  |  |  |  |

按实际结果填写。若两组差别不明显，记录现象和运行条件，不必凑出预设结论。本实验先录后放，不能用它测量边说边播的实时延迟。

### 6.3 两次识别的结果

每次可能收到多条文字回调，可按出现顺序记录，注明有没有修正。

| 次数 | 观察到的文字与变化 | 与说话内容是否一致 | 取消后的状态 / done() |
| :---: | :--- | :--- | :--- |
| 第1次 |  |  |  |
| 第2次 |  |  |  |

### 6.4 检查点

- 是否能区分采样数据、转写文字和理解提问的含义。
- 是否能解释相同时长下，三声道的数据量为什么是单声道的三倍。
- 是否能说明队列满时三种溢出策略的区别，并根据记录解释两组写入现象。
- 是否能用返回状态判断播放是否成功，以及识别任务是否已经结束。

### 6.5 思考题

- 两次说同一句话，识别结果不一样。哪些输入条件可能变了？怎样比较才更公平？
- 输入的声音块比播放快时，多出的块暂存在哪里？队列满后会怎样？挑一种 `overflow` 设定说明。
- 语音留言转写和会议实时字幕，分别适合怎样的识别方式？说出理由。
